# **Práctica 1**
## **Parte 1.**
---
### Encargado: **Marcos Bermejo Agenjo**
---


Consiste en la programación de un script que resuelva un problema de programación lineal. Se puede introducir ya en forma matricial sin necesidad de ir capturando una por una las entradas, como si de un software comercial se tratase.

Se trata de programar el algoritmo **«Símplex completo»** visto en clase. Decidirá automáticamente si empezar con Símplex primal o dual y cambiará si fuera necesario. **No hace falta considerar casos degenerados.**

---
## **Solución:**

### Objetivo y alcance

Se implementa el **algoritmo Símplex combinado** de los apuntes de la asignatura. El procedimiento decide si puede comenzar con Símplex estándar o dual y conserva la base al cambiar de método. La aplicación del solucionador al **L1-SVM** constituye la siguiente etapa de esta parte de la práctica.

Se admiten problemas de maximización o minimización con restricciones `<=` y `>=`, y variables originales no negativas. No se contemplan igualdades ni degeneración, de acuerdo con el alcance indicado en los apuntes y el enunciado.

**Limitación:** si la tabla inicial no es primal ni dual factible, el algoritmo puede devolver `"no_resuelto"`. Este estado no equivale a infactibilidad. No se incluye una Fase I general ni una transformación automática del problema al dual de programación lineal.

### Entrada del algoritmo

Un problema de programación lineal se describe mediante:

$$
\operatorname{opt}_{x\geq 0}\; c^T x,
\qquad a_i^T x\;\mathrel{\square_i}\;b_i,\quad i=1,\ldots,m,
$$

donde cada $\square_i$ es $\leq$ o $\geq$.

La función principal recibe:

| Parámetro | Significado |
| --- | --- |
| `opt` | `"max"` o `"min"` |
| `c` | Vector de $n$ coeficientes del objetivo |
| `A` | Matriz de restricciones de tamaño $m\times n$ |
| `rest` | $m$ signos: `"<="` o `">="` |
| `b` | Vector de $m$ términos independientes |

Las variables auxiliares no se incluyen en la entrada. Por ejemplo:

$$
\begin{aligned}
\max\quad & 5x_1+4x_2\\
\text{s.a.}\quad &4x_1+2x_2\leq 8\\
&x_1+x_2\leq 3\\
&x_1,x_2\geq 0.
\end{aligned}
$$

En este caso, $c=(5,4)^T$, $A=\left(\begin{smallmatrix}4&2\\1&1\end{smallmatrix}\right)$, $b=(8,3)^T$ y `rest = ["<=", "<="]`.

### Diseño de la implementación y decisiones adoptadas

**`preprocesar()`** valida las dimensiones, los tipos de restricciones y los datos numéricos. Convierte cada desigualdad `>=` en `<=` multiplicando ambos miembros por $-1$. Después incorpora una variable auxiliar por restricción. Así se obtiene una matriz ampliada $\bar A=[\widetilde A\mid I_m]$ y el tableau inicial

$$
M_0=\begin{pmatrix}
1&-\bar c^T&0\\
0&\bar A&\widetilde b
\end{pmatrix},
\qquad\bar c=(c^T,0,\ldots,0)^T.
$$

**`actualizar_matriz()`** reconstruye la tabla de cada iteración desde la matriz de referencia, como se indica en los apuntes, en lugar de encadenar operaciones de fila sobre la tabla anterior. Sea $B$ la submatriz formada por las columnas **básicas de $\bar A$** y $c_B$ el vector de coeficientes del objetivo de esas variables. Entonces:

$$
M=\begin{pmatrix}
1&c_B^TB^{-1}\\
0&B^{-1}
\end{pmatrix}M_0.
$$

Se mantiene el cálculo explícito de $B^{-1}$ por **fidelidad al desarrollo matricial del curso**. Para aplicaciones grandes sería preferible resolver sistemas lineales sin formar la inversa explícitamente.

La lista `base` contiene **índices de columnas del tableau completo**, no índices relativos a $\bar A$: la columna 0 corresponde a $z$, las columnas $1,\ldots,n$ a las variables originales y las siguientes a las auxiliares. El orden de `base` corresponde al orden de las filas de restricciones.

**`simplex_estandar()`** requiere una base primal factible y trabaja con una función objetivo de maximización. Entra una variable no básica con coste reducido negativo; sale la básica que minimiza $b_i/a_{ie}$ entre los pivotes positivos. Se detiene en el óptimo o cuando detecta no acotación o alcanza el límite de iteraciones.

**`simplex_dual()`** requiere una tabla dual factible, en la convención de minimización empleada en clase (costes reducidos no positivos). Sale la variable básica cuyo término independiente es más negativo; entra la no básica que minimiza $|r_j/a_{sj}|$ entre las columnas con $a_{sj}<0$. Se detiene al recuperar la factibilidad primal o cuando detecta infactibilidad o alcanza el límite de iteraciones.

**`simplex_completo()`** coordina las dos variantes. Antes del estándar, prepara un objetivo de maximización; antes del dual, uno de minimización. Si el dual termina correctamente, conserva su base y reconstruye la tabla desde la matriz de referencia correspondiente a la maximización. Para interpretar el resultado solo recupera las variables originales; las no básicas valen cero y el signo del objetivo se restaura cuando el problema original era de minimización.

**Decisiones de alcance y estabilidad:** se mantiene la formulación matricial de los apuntes y no se añade una Fase I general. Las comprobaciones numéricas utilizarán una tolerancia $\varepsilon=10^{-9}$ para evitar que el redondeo se interprete como un cambio real de signo. Se rechazan entradas de dimensiones inconsistentes y valores no finitos. Estas decisiones deben reflejarse también en las funciones antes de considerar finalizada la documentación.

### Flujo de resolución

```mermaid
flowchart TD
    A["simplex_completo: problema original"] --> B["preprocesar: construir M0"]
    B --> C{"¿Base inicial primal factible?"}
    C -->|Sí| D["Preparar objetivo de maximización"]
    D --> E["simplex_estandar"]
    C -->|No| F["Preparar objetivo de minimización"]
    F --> G{"¿Tabla inicial dual factible?"}
    G -->|No| H["Devolver no_resuelto"]
    G -->|Sí| I["simplex_dual"]
    I --> J{"¿Estado óptimo?"}
    J -->|No| K["Devolver estado y última matriz"]
    J -->|Sí| L["Conservar base y preparar objetivo de maximización"]
    L --> E
    E --> M{"¿Estado óptimo?"}
    M -->|No| N["Devolver estado y última matriz"]
    M -->|Sí| O["Recuperar variables originales y objetivo"]
```

La transición del dual al estándar se realiza con la **misma base** y una nueva matriz de referencia. Ambos algoritmos utilizan `actualizar_matriz()`.

| Condición de arranque | Procedimiento |
| --- | --- |
| Base primal factible | Preparar maximización y ejecutar estándar |
| Base no primal factible, pero tabla dual factible | Ejecutar dual, conservar base y pasar a estándar |
| Ninguna de las dos condiciones | Devolver `"no_resuelto"` |

La tabla resume el diagrama también cuando el visor del notebook no renderiza Mermaid.


### Salida, estados y comprobación

La interfaz de `simplex_completo()` devuelve `(x, z, estado, M_f)`:

- `x`: valores de las variables **originales**, sin auxiliares.
- `z`: valor de la función objetivo en el problema original.
- `estado`: `"optimo"`, `"no acotado"`, `"infactible"`, `"max_iter"` o `"no_resuelto"`.
- `M_f`: última tabla calculada, útil para inspección y depuración.

Los valores `x` y `z` solo se presentan como óptimos si `estado == "optimo"`; en los demás casos ambos son `None`. `"no_resuelto"` indica que no se cumple ninguna condición de arranque implementada y **no** demuestra infactibilidad. `"max_iter"` indica un límite computacional, no una clasificación matemática del problema.

Se contrastarán los resultados con problemas de solución conocida: estándar y dual, maximización y minimización, no acotación, infactibilidad y casos fuera del alcance de la tabla inicial. Además de comparar $x$ y $z$, se verificará que $c^Tx=z$ y que las restricciones originales se satisfacen dentro de la tolerancia numérica.

In [1]:
import numpy as np

In [2]:
def actualizar_matriz(M0: np.ndarray, base: list):
    n_rest = M0.shape[0] - 1
    
    # Comprobamos que la base es correcta
    if len(base) != n_rest or len(set(base)) != n_rest:
     raise ValueError('La base debe contener una variable distinta por restricción.')
    if any(not isinstance(j, (int, np.integer)) or not 1 <= j < M0.shape[1] - 1 for j in base):
        raise ValueError('Los índices de la base deben señalar columnas de variables.')

    # Extraemos la matriz básica B usando los índices de la base en M0
    B = M0[1:, base]
    
    try:
        B_inv = np.linalg.inv(B)
    except np.linalg.LinAlgError as exc:
        raise ValueError('La matriz básica es singular.')
    
    # Coeficientes de las variables básicas en la función objetivo
    c_B = -M0[0, base]
    
    # Construimos la matriz de transformación T
    fila_T = np.hstack(([1], c_B @ B_inv))
    resto_T = np.hstack((np.zeros((n_rest, 1)), B_inv))
    T = np.vstack((fila_T, resto_T))
    
    # Calculamos la nueva matriz M
    M = T @ M0
    
    return M

In [3]:
def preprocesar(opt: str, c: np.ndarray, A: np.ndarray, rest: list[str], b: np.ndarray):
    # Convertimos datos numéricos
    try:
        c = np.asarray(c, dtype=float)
        A = np.asarray(A, dtype=float)
        b = np.asarray(b, dtype=float)
    except (TypeError, ValueError) as exc:
        raise ValueError('c, A y b deben contener valores numéricos.') from exc

    # Comprobamos que la entrada es válida
    if A.ndim != 2 or b.ndim != 1 or c.ndim != 1:
        raise ValueError("Dimensiones de la entrada incorrectas.")
    if opt not in ("max", "min"):
        raise ValueError("La optimización debe ser 'max' o 'min'.")
    if any(r not in ("<=", ">=") for r in rest):
        raise ValueError("Las restricciones deben ser '<=' o '>='.")
    if len(rest) != len(b) or len(b) != len(A):
        raise ValueError("El número de restricciones no coincide.")
    if len(c) != A.shape[1]:
        raise ValueError("El número de variables no coincide.")
    if len(c) == 0 or len(b) == 0:
        raise ValueError('Debe haber al menos una variable y una restricción.')
    if not (np.all(np.isfinite(c)) and np.all(np.isfinite(A)) and np.all(np.isfinite(b))):
        raise ValueError('Los datos no pueden contener NaN ni infinitos.')
    
    
    n_rest = len(rest)
    n_var = len(c)
    c_, A_, b_ = c.copy(), A.copy(), b.copy()
        
    # Convertimos todas las restricciones a '<='
    for i in range(n_rest):
        if rest[i] == ">=":
            A_[i] *= -1
            b_[i] *= -1
            
    # Añadimos variables de holgura
    I_n = np.eye(n_rest)
    A_ = np.hstack((A_, I_n))
    c_ = np.hstack((c_, np.zeros(n_rest)))
    
    # Calculamos la matriz inicial
    renglon_z = np.hstack(([1], -c_, [0]))
    restricciones = np.hstack((np.zeros((n_rest, 1)),
                               A_,
                               b_.reshape(-1, 1)
                               ))
    M0 = np.vstack((renglon_z, restricciones))
    
    return opt, M0

In [4]:
def simplex_estandar(M0: np.ndarray, base: list = None, 
                     max_iter: int = 1000, tol: float = 1e-9):
    estado = ''
    nf, nc = M0.shape
    n_rest = nf - 1
    n_var = nc - n_rest - 2
    
    # Si no definimos la base (iteración 0)
    if base is None:
    # Calculamos los índices (comenzamos en x=(0,...,0))
        base_ = [i for i in range(n_var+1, n_rest + n_var + 1)]
    else:
        base_ = base.copy()
    
    # Actualizamos la matriz
    M = actualizar_matriz(M0, base_)
    
    it = 0
    while it < max_iter:
        no_base = [i for i in range(1,nc-1) if i not in base_]
        
        # Calculamos la variable de entrada
        mas_neg = -tol
        ent = -1
        for i in no_base:
            actual = M[0, i]
            if actual < mas_neg:
                mas_neg = actual
                ent = i
                
        # Criterio de parada: no quedan coeficientes negativos en el renglón z
        if ent == -1:
            return M, base_, 'optimo'

        # Calculamos variable de salida
        menor = float('inf')
        for j in range(1, nf):
            if M[j, ent] <= tol:
                continue
            cociente = M[j, -1] / M[j, ent]
            if cociente < menor:
                menor = cociente
                sal = j
        
        if menor == float('inf'):
            estado = 'no acotado'
            return M, base_, estado
        
        # Modificamos la posición correspondiente a la fila de la matriz de entrada,
        # ajustando el índice
        base_[sal-1] = ent
        
        # Utilizamos la matriz original con la base modificada
        M = actualizar_matriz(M0, base_)
        
        it += 1
    
    # Comprobamos si el último pivote alcanzó el óptimo.
    if np.all(M[0, 1:-1] >= -tol):
        return M, base_, 'optimo'

    return M, base_, 'max_iter'

In [5]:
matriz, base, estado = simplex_estandar(np.array([
                            [1, -5, -4, 0, 0, 0], 
                            [0, 4, 2, 1, 0, 8], 
                            [0, 1, 1, 0, 1, 3]]))
print(matriz, base, estado)

[[ 1.   0.   0.   0.5  3.  13. ]
 [ 0.   1.   0.   0.5 -1.   1. ]
 [ 0.   0.   1.  -0.5  2.   2. ]] [1, 2] optimo


In [6]:
def simplex_dual(M0: np.ndarray, base: list = None, 
                 max_iter: int = 1000, tol: float = 1e-9):
    estado = ''
    nf, nc = M0.shape
    n_rest = nf - 1
    n_var = nc - n_rest - 2
    
    # Si no definimos la base (iteración 0)
    if base is None:
    # Calculamos los índices (comenzamos en x=(0,...,0))
        base_ = [i for i in range(n_var+1, n_rest + n_var + 1)]
    else:
        base_ = base.copy()
    
    # Actualizamos la matriz
    M = actualizar_matriz(M0, base_)
    
    it = 0
    while it < max_iter:
        no_base = [i for i in range(1,nc-1) if i not in base_]
        
        # Calculamos la variable de salida
        mas_neg = -tol
        sal = -1
        for i in range(1, nf):
            actual = M[i, -1]
            if actual < mas_neg:
                mas_neg = actual
                sal = i
                
        # Criterio de parada: no quedan términos independientes negativos
        if sal == -1:
            return M, base_, 'optimo'

        # Calculamos variable de entrada
        menor = float('inf')
        for j in no_base:
            if M[sal, j] >= -tol:
                continue
            cociente = np.abs(M[0, j] / M[sal, j])
            if cociente < menor:
                menor = cociente
                ent = j
        
        if menor == float('inf'):
            estado = 'infactible'
            return M, base_, estado
        
        # Modificamos la posición correspondiente a la fila de la matriz de entrada,
        # ajustando el índice
        base_[sal-1] = ent
        
        # Utilizamos la matriz original con la base modificada
        M = actualizar_matriz(M0, base_)
        
        it += 1
    
    # Comprobamos si el último pivote recuperó la factibilidad primal.
    if np.all(M[1:, -1] >= -tol):
        return M, base_, 'optimo'

    return M, base_, 'max_iter'

In [7]:
matriz, base, estado = simplex_dual(np.array([
                            [1, -8, -3, 0, 0, 0], 
                            [0, -4, -1, 1, 0, -5], 
                            [0, -2, -1, 0, 1, -4]]))
print(matriz, base, estado)

[[ 1.   0.   0.  -1.  -2.  13. ]
 [ 0.   1.   0.  -0.5  0.5  0.5]
 [ 0.   0.   1.   1.  -2.   3. ]] [1, 2] optimo


In [ ]:
def simplex_completo(opt, c, A, rest, b, tol: float = 1e-9):
    opt_, M0 = preprocesar(opt, c, A, rest, b)
    # Trabajamos con una copia de M0
    M = M0.copy()
    
    # ¿Se cumple factibilidad primal?
    # Si sí, aplicamos estándar
    if np.all(M[1:, -1] >= -tol):
        # Queremos un problema de maximización
        if opt_ == "min":
            M[0, 1:] *= -1
            opt_ = "max"
        M1, base, estado = simplex_estandar(M, tol=tol)
    else: # Si no, cambiamos de max a min y aplicamos dual
        # Queremos un problema de minimización
        if opt_ == "max":
            M[0, 1:] *= -1
            opt_ = "min"
        if np.all(M[0, 1:-1] <= tol): # ¿Se cumple factibilidad dual?
            M, base, estado = simplex_dual(M, tol=tol)
            # Si el problema original era de minimización,
            # lo cambiamos a maximización para usar estándar
            if opt == "min":
                M0[0, 1:] *= -1
            # Pasamos a estándar si devuelve un resultado válido
            if estado == "optimo":
                M1, base, estado = simplex_estandar(M0, base, tol=tol)
            else:
                return None, None, estado, M
                
        else:
            return None, None, "no_resuelto", M
    
    # Si encontramos una solución óptima, interpretamos los resultados
    if estado == "optimo":
        x = np.zeros(len(c))
        for i, var_bas in enumerate(base):
            if var_bas <= len(c):
                x[var_bas-1] = M1[i+1, -1]

        z = M1[0, -1]
        if opt == "min":
            z *= -1
            
    else:
        return None, None, estado, M1
    
    return x, z, estado, M1

### Validación del algoritmo

Para comprobar el funcionamiento de la implementación se ha preparado una batería de pruebas que contempla problemas de maximización y minimización, el uso de los métodos Símplex primal y dual, distintas condiciones de terminación, tolerancias numéricas y validación de entradas.

**Uso de inteligencia artificial:** la batería de pruebas se ha elaborado con ayuda de ChatGPT (OpenAI, GPT-6). Su finalidad es contrastar los resultados obtenidos con soluciones conocidas y detectar posibles errores en la implementación.

Estas pruebas permiten verificar el comportamiento del algoritmo en los casos considerados, pero no constituyen una demostración de su corrección general.

In [9]:

import numpy as np

# Pruebas del Símplex completo
casos = [
    ("Máximo estándar", "max", [5, 4], [[4, 2], [1, 1]],
     ["<=", "<="], [8, 3], "optimo", [1, 2], 13),

    ("Mínimo estándar", "min", [-5, -4], [[4, 2], [1, 1]],
     ["<=", "<="], [8, 3], "optimo", [1, 2], -13),

    ("Máximo con varios pivotes", "max", [3, 2],
     [[1, 1], [1, 0], [0, 1]],
     ["<=", "<=", "<="], [4, 2, 3], "optimo", [2, 2], 10),

    ("Máximo con un pivote", "max", [1], [[1]],
     ["<="], [1], "optimo", [1], 1),

    ("Óptimo en el origen", "max", [-1], [[1]],
     ["<="], [2], "optimo", [0], 0),

    ("Mínimo con inicio dual", "min", [8, 3],
     [[4, 1], [2, 1]],
     [">=", ">="], [5, 4], "optimo", [0.5, 3], 13),

    ("Mínimo dual con restricciones mixtas", "min", [1],
     [[1], [1]], [">=", "<="], [1, 3], "optimo", [1], 1),

    ("Máximo con inicio dual", "max", [-1],
     [[1], [1]], [">=", "<="], [1, 3], "optimo", [1], -1),

    ("No acotado", "max", [1], [[-1]],
     ["<="], [1], "no acotado", None, None),

    ("Infactible", "max", [-1], [[1]],
     ["<="], [-1], "infactible", None, None),

    ("Sin factibilidad inicial primal ni dual (max)", "max", [1],
     [[1], [1]], [">=", "<="], [1, 3], "no_resuelto", None, None),

    ("Sin factibilidad inicial primal ni dual (min)", "min", [-1],
     [[1], [1]], [">=", "<="], [1, 3], "no_resuelto", None, None),
]

fallos = []
correctas = 0
atol = 1e-8

def comprobar(nombre, prueba):
    global correctas
    try:
        prueba()
        correctas += 1
        print(f"OK   {nombre}")
    except Exception as e:
        fallos.append((nombre, str(e)))
        print(f"FALLO {nombre}: {e}")


# 1. Resultados conocidos y condiciones de factibilidad
for nombre, opt, c, A, rest, b, esperado, x_esp, z_esp in casos:

    def prueba():
        c_, A_, b_ = map(lambda v: np.asarray(v, dtype=float), (c, A, b))
        x, z, estado, M = simplex_completo(opt, c_, A_, rest, b_)

        assert estado == esperado, f"Estado {estado!r}, esperado {esperado!r}"
        assert isinstance(M, np.ndarray), "No se ha devuelto un tableau"

        if esperado == "optimo":
            assert x is not None and z is not None
            np.testing.assert_allclose(x, x_esp, atol=atol, rtol=0)
            np.testing.assert_allclose(z, z_esp, atol=atol, rtol=0)

            # No negatividad de las variables originales
            assert np.all(x >= -atol), "Variables originales negativas"

            # Comprobación de todas las restricciones originales
            lhs = A_ @ x
            for i, signo in enumerate(rest):
                if signo == "<=":
                    assert lhs[i] <= b_[i] + atol, f"Restricción {i+1} incumplida"
                else:
                    assert lhs[i] >= b_[i] - atol, f"Restricción {i+1} incumplida"

            # Coherencia entre la solución y el valor objetivo
            np.testing.assert_allclose(z, c_ @ x, atol=atol, rtol=0)
        else:
            assert x is None and z is None, "Se devolvió solución sin óptimo"

    comprobar(nombre, prueba)


# 2. Comprobación del último pivote permitido
def prueba_max_iter_estandar():
    _, M0 = preprocesar("max", np.array([1.]),
                        np.array([[1.]]), ["<="], np.array([1.]))
    M, base, estado = simplex_estandar(M0, max_iter=1)
    assert estado == "optimo", f"Estado obtenido: {estado}"
    np.testing.assert_allclose(M[0, -1], 1, atol=atol)

comprobar("max_iter: último pivote estándar", prueba_max_iter_estandar)


def prueba_max_iter_dual():
    _, M0 = preprocesar("min", np.array([1.]),
                        np.array([[1.]]), [">="], np.array([1.]))
    M, base, estado = simplex_dual(M0, max_iter=1)
    assert estado == "optimo", f"Estado obtenido: {estado}"
    np.testing.assert_allclose(M[1, -1], 1, atol=atol)

comprobar("max_iter: último pivote dual", prueba_max_iter_dual)


# 3. Tolerancias numéricas: residuos cercanos a cero
def prueba_tolerancia_estandar():
    M0 = np.array([[1., -1e-12, 0., 0.],
                   [0., 1., 1., 1.]])
    M, base, estado = simplex_estandar(M0)
    assert estado == "optimo"
    assert base == [2], "Se ha pivotado por un residuo numérico"

comprobar("Tolerancia estándar", prueba_tolerancia_estandar)


def prueba_tolerancia_dual():
    M0 = np.array([[1., -1., 0., 0.],
                   [0., -1., 1., -1e-12]])
    M, base, estado = simplex_dual(M0)
    assert estado == "optimo"
    assert base == [2], "Se ha pivotado por un residuo numérico"

comprobar("Tolerancia dual", prueba_tolerancia_dual)


# 4. Validación de entradas
def prueba_listas():
    x, z, estado, M = simplex_completo(
        "max", [1], [[1]], ["<="], [2]
    )
    assert estado == "optimo"
    np.testing.assert_allclose(x, [2], atol=atol)
    np.testing.assert_allclose(z, 2, atol=atol)

comprobar("Entrada mediante listas Python", prueba_listas)


entradas_invalidas = [
    ("Sentido inválido", "otro", [1], [[1]], ["<="], [2]),
    ("Restricción inválida", "max", [1], [[1]], ["="], [2]),
    ("Dimensión incorrecta", "max", [1, 2], [[1]], ["<="], [2]),
    ("NaN en coeficientes", "max", [np.nan], [[1]], ["<="], [2]),
    ("Infinito en restricciones", "max", [1], [[np.inf]], ["<="], [2]),
]

for nombre, opt, c, A, rest, b in entradas_invalidas:
    def prueba():
        try:
            simplex_completo(opt, c, A, rest, b)
        except (ValueError, TypeError):
            return
        raise AssertionError("La entrada inválida no fue rechazada")

    comprobar(nombre, prueba)


# 5. Resumen
total = correctas + len(fallos)
print(f"\nRESULTADO: {correctas}/{total} comprobaciones superadas")

if fallos:
    print("\nComprobaciones pendientes:")
    for nombre, error in fallos:
        print(f" - {nombre}: {error}")
    raise AssertionError(f"{len(fallos)} comprobaciones fallidas")
else:
    print("Todas las comprobaciones han sido superadas.")


OK   Máximo estándar
OK   Mínimo estándar
OK   Máximo con varios pivotes
OK   Máximo con un pivote
OK   Óptimo en el origen
OK   Mínimo con inicio dual
OK   Mínimo dual con restricciones mixtas
OK   Máximo con inicio dual
OK   No acotado
OK   Infactible
OK   Sin factibilidad inicial primal ni dual (max)
OK   Sin factibilidad inicial primal ni dual (min)
OK   max_iter: último pivote estándar
OK   max_iter: último pivote dual
OK   Tolerancia estándar
OK   Tolerancia dual
OK   Entrada mediante listas Python
OK   Sentido inválido
OK   Restricción inválida
OK   Dimensión incorrecta
OK   NaN en coeficientes
OK   Infinito en restricciones

RESULTADO: 22/22 comprobaciones superadas
Todas las comprobaciones han sido superadas.
